# 第 11 章:推理工程 —— API 服务 / 流式 / 工具调用

前面的章节都在关心一件事:**怎么训练出一个模型**。从预训练(第 6 章)到 SFT(第 9 章)、LoRA(第 10 章),我们已经能把一个 64M 的模型教到能对话。

但训练完的模型只是 `.pth` 文件 —— 一个 PyTorch `state_dict`。它不能被任何外部程序调用,不能接入前端,不能被运维监控。

本章解决最后一个工程问题:**如何把模型变成一个可用的服务**。

> 本章的代码全部对应 `scripts/serve_openai_api.py`(252 行)和 `scripts/convert_model.py`(144 行)。所有行号基于 minimind 源码。

## 11.0 本章路线图

推理工程的核心问题可以拆成四层:

| 层 | 问题 | 对应源码 |
|---|---|---|
| API 协议 | 外部程序怎么调用? | `serve_openai_api.py:178` `/v1/chat/completions` |
| 流式输出 | 生成太慢怎么办? | `serve_openai_api.py:71-80` `CustomStreamer` |
| 输出解析 | `<think>` 和 `<tool_call>` 怎么处理? | `serve_openai_api.py:83-102` `parse_response` |
| 格式转换 | vllm / ollama / llama.cpp 怎么部署? | `convert_model.py:40-96` |

> 一句话总结:**OpenAI API 格式是事实标准,流式用 SSE,模型用 HF transformers 格式。**

```
训练侧 (.pth)                    部署侧
┌─────────────┐    convert_model    ┌──────────────────┐
│ torch 权重   │ ──────────────────→ │ HF transformers  │ ──→ vllm / ollama
│ (MiniMind)  │    LoRA merge       │ (Qwen3 结构)      │ ──→ llama.cpp (GGUF)
└─────────────┘                     └────────┬─────────┘
       ↑                                      │
   trainer 训练                          serve_openai_api
                                             │
                                      FastAPI :8998
                                             │
                                   /v1/chat/completions
                                             │
                                      前端 / Agent / curl
```

&nbsp;

---

## Part 1:从 CLI 到 API

### 11.1 为什么需要 API 服务

第 1 章我们用 `eval_llm.py` 做命令行推理:

```python
# eval_llm.py 的核心 —— 命令行交互
while True:
    user = input("💬: ")
    response = model.generate(...)
    print(f"🧠: {response}")
```

这对测试足够,但生产环境需要:

| 需求 | CLI (eval_llm.py) | API 服务 |
|---|---|---|
| 调用方式 | 命令行 `input()` | HTTP POST |
| 并发 | 单进程,一次一个 | 多请求并发 |
| 集成 | 无法接入外部 | 任何语言都能调 |
| 协议 | 无 | OpenAI 兼容 |
| 流式 | 无 | SSE streaming |

**关键决策:采用 OpenAI API 格式。**

为什么?因为整个 AI 生态 —— LangChain、AutoGen、Cursor、各种 Agent 框架 —— 都已经适配了 OpenAI 的 `/v1/chat/completions` 接口。只要你的服务器兼容这个格式,就能**零成本接入**所有现有工具。

> 这是「事实标准」的力量:不是最好,但所有人都在用。minimind 选择 OpenAI 格式,意味着 `eval_toolcall.py`、`eval_model.py` 等评估脚本可以直接用 `openai` Python SDK 调用,无需自己写 client。

&nbsp;

---

## 11.2 FastAPI 服务器骨架

`serve_openai_api.py` 用 FastAPI 实现了一个完整的 OpenAI 兼容服务。核心就一个路由:`POST /v1/chat/completions`。

先看请求 schema(`ChatRequest`,第 50-59 行):

```python
class ChatRequest(BaseModel):
    model: str
    messages: list
    temperature: float = 0.7
    top_p: float = 0.92
    max_tokens: int = 8192
    stream: bool = True              # 默认流式
    tools: list = Field(default_factory=list)
    open_thinking: bool = False      # minimind 扩展字段
    chat_template_kwargs: dict = None
```

和 OpenAI 官方 schema 几乎一样,多了两个字段:
- `open_thinking`:控制是否输出 `<think>` 推理过程
- `chat_template_kwargs`:透传给 chat template 的额外参数

下面我们用最少的代码搭一个能跑的 server:

In [ ]:
# 最小 FastAPI server —— 只做演示,实际用 serve_openai_api.py
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import List, Optional
import json, time, uuid

app = FastAPI(title="MiniMind API (demo)")

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str = "minimind"
    messages: List[Message]
    temperature: float = 0.7
    max_tokens: int = 8192
    stream: bool = True

@app.post("/v1/chat/completions")
async def chat(req: ChatRequest):
    """OpenAI 兼容的 chat completions 端点"""
    # 构造 OpenAI 格式响应
    resp = {
        "id": f"chatcmpl-{uuid.uuid4().hex[:8]}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [{
            "index": 0,
            "message": {
                "role": "assistant",
                "content": f"[demo] 收到 {len(req.messages)} 条消息"
            },
            "finish_reason": "stop"
        }]
    }
    return resp

# 说明:这只是一个不加载模型的骨架 demo
# 完整实现见 scripts/serve_openai_api.py —— 会调用 model.generate()
print("启动方式: uvicorn <filename>:app --port 8998")
print("调用方式: curl -X POST http://localhost:8998/v1/chat/completions -H 'Content-Type: application/json' -d '{\"model\":\"minimind\",\"messages\":[{\"role\":\"user\",\"content\":\"你好\"}]}'")

### 非流式响应格式

当 `stream=False` 时,返回完整的 JSON 响应(对应 `serve_openai_api.py:193-232`):

```json
{
  "id": "chatcmpl-1700000000",
  "object": "chat.completion",
  "created": 1700000000,
  "model": "minimind",
  "choices": [{
    "index": 0,
    "message": {
      "role": "assistant",
      "content": "你好!我是 MiniMind。",
      "reasoning_content": "用户在打招呼...",   // ← minimind 扩展
      "tool_calls": [...]                        // ← 有工具调用时出现
    },
    "finish_reason": "stop"    // 或 "tool_calls"
  }]
}
```

注意三个细节:
1. `finish_reason` 有两种值:`"stop"`(正常结束)和 `"tool_calls"`(模型想调用工具)
2. `reasoning_content` 是 minimind 对 OpenAI 格式的扩展 —— 把 `<think>` 内容单独放一个字段
3. `content` 是剥离了 `<think>` 和 `<tool_call>` 标签后的纯文本回答

> OpenAI 官方后来在 o1/o3 系列中也引入了 `reasoning_content` 字段。minimind 的实现与这个方向一致。

&nbsp;

---

## 11.3 流式输出:TextStreamer → Queue → SSE

### 为什么需要流式

LLM 生成是**自回归**的 —— 一个 token 一个 token 地吐。一个 200 token 的回答,如果等全部生成完再返回,用户要等 5-10 秒看到空白屏幕。

流式输出的价值:**用户看到第一个 token 的等待时间从「全部生成完」降到「生成出第一个 token」(TTFT)**。

```
非流式:  [============ 等 5s ============] 你好!我是 MiniMind...
流式:    你 | 好 | ! | 我 | 是 | MiniMind | ...
         ↑ 100ms 就出来
```

### 三层管道

minimind 的流式实现用了三层管道(对应 `serve_openai_api.py:71-80, 105-172`):

```
model.generate()          TextStreamer              Queue              StreamingResponse
  │  逐 token 生成          │  decode → text          │  线程安全队列       │  SSE 格式输出
  │ ──────────────────→    │ ──────────────────→    │ ──────────────→  │ ──────────────→ client
  │                         CustomStreamer            queue.put(text)     "data: {...}\n\n"
```

第一层:`CustomStreamer`(第 71-80 行)继承 transformers 的 `TextStreamer`,把每个 decode 出来的 text 片段塞进队列:

In [ ]:
from queue import Queue
from transformers import TextStreamer

# 对应 serve_openai_api.py:71-80
class CustomStreamer(TextStreamer):
    def __init__(self, tokenizer, queue):
        super().__init__(tokenizer, skip_prompt=True, skip_special_tokens=True)
        self.queue = queue

    def on_finalized_text(self, text: str, stream_end: bool = False):
        self.queue.put(text)        # 每个 text 片段入队
        if stream_end:
            self.queue.put(None)    # None 是结束信号

# 验证:模拟 tokenizer 的 finalize 回调
queue = Queue()
streamer = CustomStreamer(None, queue)  # tokenizer=None 仅演示
streamer.on_finalized_text("你好")
streamer.on_finalized_text("我是")
streamer.on_finalized_text("MiniMind", stream_end=True)

# 从队列消费
chunks = []
while True:
    item = queue.get()
    if item is None:
        break
    chunks.append(item)
print(f"队列消费到: {chunks}")
print(f"拼合: {''.join(chunks)}")

第二层:`model.generate()` 在子线程跑,`CustomStreamer` 逐块 push,主线程从队列消费(第 105-130 行):

In [ ]:
import threading
from queue import Queue

# 模拟 generate_stream_response 的核心结构 (serve_openai_api.py:113-137)
queue = Queue()

def fake_generate(queue):
    """子线程:模拟 model.generate + streamer"""
    for chunk in ["<think>", "让我想想", "</think>", "你好", "!"]:
        queue.put(chunk)
    queue.put(None)  # 结束信号

# 启动子线程
thread = threading.Thread(target=fake_generate, args=(queue,))
thread.start()

# 主线程:消费队列,生成 SSE chunk
full_text = ""
while True:
    text = queue.get()
    if text is None:
        break
    full_text += text
    print(f"  收到: {text!r:15s} → 累积: {full_text!r}")

thread.join()
print(f"\n最终文本: {full_text}")

第三层:SSE(Server-Sent Events)格式。每个 chunk 以 `data: ` 前缀 + JSON + `\n\n` 结尾:

```
data: {"choices":[{"delta":{"content":"你"}}]}

data: {"choices":[{"delta":{"content":"好"}}]}

data: {"choices":[{"delta":{"content":"!"}}]}

data: {"choices":[{"delta":{},"finish_reason":"stop"}]}
```

对应代码(第 183-191 行):

In [ ]:
# 对应 serve_openai_api.py:181-192 的 SSE 封装
def make_sse_chunk(delta: dict, finish_reason=None):
    """构造一个 SSE chunk"""
    chunk = {"choices": [{"delta": delta}]}
    if finish_reason:
        chunk["choices"][0]["finish_reason"] = finish_reason
    return f"data: {__import__('json').dumps(chunk, ensure_ascii=False)}\n\n"

# 模拟流式输出
print("=== SSE 流 ===")
for tok in ["你", "好", "!", "我是", "MiniMind"]:
    print(make_sse_chunk({"content": tok}), end="")
print(make_sse_chunk({}, finish_reason="stop"))

print("\n=== 对应的非流式响应 ===")
final_resp = {
    "id": "chatcmpl-xxx",
    "choices": [{"message": {"role": "assistant", "content": "你好!我是MiniMind"}, "finish_reason": "stop"}]
}
print(__import__('json').dumps(final_resp, ensure_ascii=False, indent=2))

注意 SSE 的三个关键格式规则:
1. 每个 chunk 以 `data: ` 开头
2. 每个 chunk 以 `\n\n` 结尾(两个换行 —— HTTP chunk 分隔)
3. 最后一个 chunk 的 `finish_reason` 非 null(告诉客户端「结束了」)

> 流式的 delta 用 `content` 字段增量传输;非流式的 message 用 `content` 字段传完整文本。字段名不同,这是 OpenAI 规范的规定。

&nbsp;

---

## Part 2:输出解析

### 11.4 `<think>` 解析:reasoning_content

模型在开启 thinking 模式时,输出格式是:

```
<think>用户在问好,我应该礼貌回复...</think>
你好!我是 MiniMind。
```

`<think>` 标签内的内容是模型的「内心独白」(reasoning chain),不应该展示给最终用户。`parse_response`(第 83-102 行)负责把它们拆开:

In [ ]:
import re

# 对应 serve_openai_api.py:83-102 的 <think> 解析
def parse_think(text):
    """提取 <think>...</think> 中的推理内容"""
    reasoning_content = None
    # 情况 1: 完整的 <think>...</think>
    think_match = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
    if think_match:
        reasoning_content = think_match.group(1).strip()
        text = re.sub(r'<think>.*?</think>\s*', '', text, flags=re.DOTALL)
    # 情况 2: 只有 </think> (开头缺少 <think>)
    elif '</think>' in text:
        parts = text.split('</think>', 1)
        reasoning_content = parts[0].strip()
        text = parts[1].strip() if len(parts) > 1 else ''
    return text.strip(), reasoning_content

# 测试:完整 think 标签
raw = "<think>用户在问好,我应该礼貌回复</think>\n你好!我是 MiniMind。"
content, reasoning = parse_think(raw)
print(f"=== 情况 1: 完整标签 ===")
print(f"reasoning_content: {reasoning}")
print(f"content:           {content}")

# 测试:缺少 <think> 开头
raw2 = "让我想想这个问题的答案\n答案是42"
content2, reasoning2 = parse_think(raw2)
print(f"\n=== 情况 2: 缺少 <think> (只有 </think>) ===")
raw2_full = "让我想想\n答案是42"
# 模拟模型输出只有 </think> 没有开头的情况
raw2_with_close = "思考过程</think>答案是42"
content2, reasoning2 = parse_think(raw2_with_close)
print(f"reasoning_content: {reasoning}")
print(f"content:           {content}")

为什么需要处理「缺少 `<think>` 开头」的情况(第 89-92 行)?因为流式生成时,`<think>` 标签可能被截断 —— 模型生成到一半,第一个 chunk 里可能没有完整标签。`parse_response` 用 `split('</think>')` 做容错处理。

> 对应到 API 响应:`reasoning_content` 放进 `message.reasoning_content`,`content` 放进 `message.content`。两个字段分离,前端可以选择是否展示推理过程。

### 11.5 `<tool_call>` 解析:tool_calls

模型在检测到需要调用工具时,输出格式是:

```
我来帮你查一下天气。

<tool_call>
{"name": "get_current_weather", "arguments": {"location": "北京"}}
</tool_call>
```

`parse_response` 的后半部分(第 93-101 行)负责提取:

In [ ]:
import re, json, time

# 对应 serve_openai_api.py:93-102 的 <tool_call> 解析
def parse_tool_calls(text):
    """提取 <tool_call>...</tool_call> 中的工具调用"""
    tool_calls = []
    for i, m in enumerate(re.findall(r'<tool_call>(.*?)</tool_call>', text, re.DOTALL)):
        try:
            call = json.loads(m.strip())
            tool_calls.append({
                "id": f"call_{int(time.time())}_{i}",
                "type": "function",
                "function": {
                    "name": call.get("name", ""),
                    "arguments": json.dumps(call.get("arguments", {}), ensure_ascii=False)
                }
            })
        except Exception:
            pass  # JSON 解析失败则跳过
    if tool_calls:
        text = re.sub(r'<tool_call>.*?</tool_call>', '', text, flags=re.DOTALL)
    return text.strip(), tool_calls or None

# 测试:单个工具调用
raw = '我来帮你查天气。\n<tool_call>\n{"name": "get_current_weather", "arguments": {"location": "北京"}}\n</tool_call>'
content, tool_calls = parse_tool_calls(raw)
print(f"=== 单个工具调用 ===")
print(f"content:    {content}")
print(f"tool_calls: {json.dumps(tool_calls, ensure_ascii=False, indent=2)}")

# 测试:多个工具调用
raw2 = '<tool_call>\n{"name": "get_current_time", "arguments": {}}\n</tool_call>\n<tool_call>\n{"name": "calculate_math", "arguments": {"expression": "2+2"}}\n</tool_call>'
content2, tool_calls2 = parse_tool_calls(raw2)
print(f"\n=== 多个工具调用 ===")
print(f"content:    '{content2}'")
print(f"tool_calls 数量: {len(tool_calls2)}")
for tc in tool_calls2:
    print(f"  - {tc['function']['name']}({tc['function']['arguments']})")

注意 `arguments` 字段被 `json.dumps` 序列化成字符串。这是 OpenAI 规范的要求:`arguments` 必须是 **JSON 字符串**,不是 JSON 对象。这样设计是为了支持流式增量传输(拼接字符串比合并 JSON 对象简单)。

### 完整的 parse_response

把 `<think>` 和 `<tool_call>` 解析合在一起,就是 `serve_openai_api.py:83-102` 的完整 `parse_response`:

In [ ]:
import re, json, time

# 完整对应 serve_openai_api.py:83-102
def parse_response(text):
    """解析模型输出:分离 <think>, <tool_call>, 正文"""
    # === 1. 解析 <think> ===
    reasoning_content = None
    think_match = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
    if think_match:
        reasoning_content = think_match.group(1).strip()
        text = re.sub(r'<think>.*?</think>\s*', '', text, flags=re.DOTALL)
    elif '</think>' in text:
        parts = text.split('</think>', 1)
        reasoning_content = parts[0].strip()
        text = parts[1].strip() if len(parts) > 1 else ''

    # === 2. 解析 <tool_call> ===
    tool_calls = []
    for i, m in enumerate(re.findall(r'<tool_call>(.*?)</tool_call>', text, re.DOTALL)):
        try:
            call = json.loads(m.strip())
            tool_calls.append({
                "id": f"call_{int(time.time())}_{i}",
                "type": "function",
                "function": {
                    "name": call.get("name", ""),
                    "arguments": json.dumps(call.get("arguments", {}), ensure_ascii=False)
                }
            })
        except Exception:
            pass
    if tool_calls:
        text = re.sub(r'<tool_call>.*?</tool_call>', '', text, flags=re.DOTALL)

    return text.strip(), reasoning_content, tool_calls or None

# 综合测试:thinking + tool_call + 正文
raw = (
    "<think>用户想知道天气,我需要调用 get_current_weather 工具</think>\n"
    "好的,我来帮你查一下北京的天气。\n"
    '<tool_call>\n{"name": "get_current_weather", "arguments": {"location": "北京"}}\n</tool_call>'
)

content, reasoning, tool_calls = parse_response(raw)
print("=== 完整 parse_response 测试 ===")
print(f"reasoning_content: {reasoning}")
print(f"content:           {content}")
print(f"tool_calls:        {json.dumps(tool_calls, ensure_ascii=False, indent=2)}")
print(f"\n→ finish_reason 应为: {'tool_calls' if tool_calls else 'stop'}")

&nbsp;

---

## 11.6 open_thinking 参数

不是每次对话都需要思考过程。`ChatRequest` 中的 `open_thinking` 参数控制是否在 chat template 中注入 `<think>` 标签(第 61-68 行):

In [ ]:
# 对应 serve_openai_api.py:50-68 的 open_thinking 兼容逻辑
class MockChatRequest:
    def __init__(self, open_thinking=False, chat_template_kwargs=None):
        self.open_thinking = open_thinking
        self.chat_template_kwargs = chat_template_kwargs

    def get_open_thinking(self):
        """兼容多种方式开启 thinking"""
        if self.open_thinking:
            return True
        if self.chat_template_kwargs:
            return self.chat_template_kwargs.get('open_thinking', False) or \
                   self.chat_template_kwargs.get('enable_thinking', False)
        return False

# 三种开启方式都兼容
cases = [
    ("直接 open_thinking=True",     MockChatRequest(open_thinking=True)),
    ("chat_template_kwargs.open_thinking", MockChatRequest(chat_template_kwargs={"open_thinking": True})),
    ("chat_template_kwargs.enable_thinking", MockChatRequest(chat_template_kwargs={"enable_thinking": True})),
    ("全部关闭",                      MockChatRequest()),
]
for label, req in cases:
    print(f"  {label:45s} → get_open_thinking() = {req.get_open_thinking()}")

`get_open_thinking()` 的返回值会传给 `tokenizer.apply_chat_template(..., open_thinking=open_thinking)`,控制 chat template 中是否包含 `<think>` 开头标签。

这种多字段兼容设计的原因:`open_thinking` 是 minimind 自己的字段,而 `enable_thinking` 是 Qwen3 / vllm 生态的约定字段。两者都支持,确保无论从哪个客户端调用都能工作。

> 对应源码:流式路径 `serve_openai_api.py:107`,非流式路径 `serve_openai_api.py:199`。两条路径都调用了 `request.get_open_thinking()`。

&nbsp;

---

## Part 3:模型格式转换

### 11.7 为什么需要格式转换

训练时,模型是 minimind 自定义的 `MiniMindForCausalLM`,保存为 PyTorch `.pth` 文件。但**部署生态几乎都只认 HuggingFace transformers 格式**:

| 部署框架 | 需要的格式 | 原因 |
|---|---|---|
| vllm | HF transformers | 用 `AutoModelForCausalLM.from_pretrained` 加载 |
| ollama | HF / GGUF | 先转 HF,再转 GGUF |
| llama.cpp | GGUF | 从 HF 格式转换 |
| transformers | HF transformers | 原生格式 |
| minimind 自身 | `.pth` | 直接用 `torch.load` |

`convert_model.py` 的核心任务:把 `.pth` → HF transformers,并且映射到 `Qwen3Config` 结构。

### 为什么映射到 Qwen3?

minimind 的模型架构和 Qwen3 几乎一模一样(GQA + RoPE + RMSNorm + SwiGLU)。直接映射到 `Qwen3Config` 有两个好处:
1. **vllm / TensorRT-LLM 有 Qwen3 的专用优化 kernel**,性能远超自定义模型
2. 不需要 `trust_remote_code=True`,部署更安全

In [ ]:
# 对应 convert_model.py:43-81 的配置映射逻辑
# 展示 MiniMind config → Qwen3 config 的字段对应关系

lm_config = {
    "vocab_size": 6400,
    "hidden_size": 768,
    "intermediate_size": 2048,
    "num_hidden_layers": 8,
    "num_attention_heads": 8,
    "num_key_value_heads": 4,         # GQA
    "max_position_embeddings": 8192,
    "rms_norm_eps": 1e-5,
    "rope_theta": 10000.0,
    "tie_word_embeddings": True,
}

# 映射到 Qwen3Config 的字段 (convert_model.py:43-55)
common_config = {
    "vocab_size": lm_config["vocab_size"],
    "hidden_size": lm_config["hidden_size"],
    "intermediate_size": lm_config["intermediate_size"],
    "num_hidden_layers": lm_config["num_hidden_layers"],
    "num_attention_heads": lm_config["num_attention_heads"],
    "num_key_value_heads": lm_config["num_key_value_heads"],
    "head_dim": lm_config["hidden_size"] // lm_config["num_attention_heads"],  # 96
    "max_position_embeddings": lm_config["max_position_embeddings"],
    "rms_norm_eps": lm_config["rms_norm_eps"],
    "rope_theta": lm_config["rope_theta"],
    "tie_word_embeddings": lm_config["tie_word_embeddings"],
}

print("=== Qwen3Config 字段映射 ===")
for k, v in common_config.items():
    print(f"  {k:30s} = {v}")

print(f"\n  head_dim = hidden_size // num_attention_heads = {lm_config['hidden_size']} // {lm_config['num_attention_heads']} = {common_config['head_dim']}")
print(f"\n  → 保存为 Qwen3ForCausalLM,vllm 会用 Qwen3 专用 kernel 加载")

### MoE Expert Stacking

对于 MoE 模型,有一个额外的挑战:`MiniMindMoE` 把每个 expert 存为独立的 Linear 层,但 `Qwen3MoeForCausalLM` 把所有 expert **stack 成一个大的 tensor**(第 72-79 行):

```python
# 转换前 (MiniMind): 逐个 expert 独立存储
layers.0.mlp.experts.0.gate_proj.weight   # (2048, 768)
layers.0.mlp.experts.1.gate_proj.weight   # (2048, 768)
...
layers.0.mlp.experts.7.gate_proj.weight   # (2048, 768)

# 转换后 (Qwen3MoE): stack 成一个 tensor
layers.0.mlp.experts.gate_up_proj         # (8, 4096, 768)  ← 8 experts 合并
layers.0.mlp.experts.down_proj            # (8, 768, 2048)
```

这样做的目的:让 vllm 能用 **batched expert kernel** 同时计算所有 expert,大幅提升 MoE 推理速度。

In [ ]:
import torch

# 模拟 convert_model.py:74-78 的 MoE expert stacking
num_experts = 8
hidden_size = 768
intermediate_size = 2048

# 转换前:8 个独立的 expert 权重
experts_gate = [torch.randn(intermediate_size, hidden_size) for _ in range(num_experts)]
experts_up   = [torch.randn(intermediate_size, hidden_size) for _ in range(num_experts)]
experts_down = [torch.randn(hidden_size, intermediate_size) for _ in range(num_experts)]

print(f"=== 转换前: {num_experts} 个独立 expert ===")
print(f"每个 expert: gate_proj {tuple(experts_gate[0].shape)} × 3 组")

# gate_up_proj: stack gate 和 up,然后按 expert 维度合并 (convert_model.py:77)
gate_up_proj = torch.cat([
    torch.stack(experts_gate),  # (8, 2048, 768)
    torch.stack(experts_up),    # (8, 2048, 768)
], dim=1)                       # (8, 4096, 768)

# down_proj: 直接 stack (convert_model.py:78)
down_proj = torch.stack(experts_down)  # (8, 768, 2048)

print(f"\n=== 转换后: stacked tensor ===")
print(f"experts.gate_up_proj: {tuple(gate_up_proj.shape)}  ← (num_experts, 2×intermediate, hidden)")
print(f"experts.down_proj:    {tuple(down_proj.shape)}  ← (num_experts, hidden, intermediate)")
print(f"\n  → vllm 可以用一次 batched matmul 计算所有 expert")

&nbsp;

---

## 11.8 LoRA 合并

训练完 LoRA 后,得到的是一个极小的「补丁」文件(~0.8 MB)。部署时不能让推理引擎实时加载 base + LoRA(性能差),需要先把它们**合并**成一个完整模型。

`convert_merge_base_lora`(第 105-112 行)做这件事:

In [ ]:
# 对应 convert_model.py:105-112 的 LoRA 合并逻辑 (伪代码展示流程)
print("""
=== LoRA 合并流程 (convert_model.py:105-112) ===

输入:
  base_torch_path:  ../out/full_sft_768.pth        # 基础模型 (~131 MB)
  lora_path:        ../out/lora_medical_768.pth     # LoRA 补丁 (~0.8 MB)

步骤:
  1. lm_model = MiniMindForCausalLM(config)          # 创建空模型
  2. lm_model.load_state_dict(torch.load(base_path))  # 加载 base 权重
  3. apply_lora(lm_model)                             # 挂载 LoRA 模块 (B=0)
  4. merge_lora(lm_model, lora_path, merged_path)     # W += BA, 保存

  merge_lora 内部:
    for each LoRA layer:
        load_lora(layer, lora_path)     # 加载训练好的 A, B
        W_merged = W + B @ A            # 折叠 LoRA 进原始权重
    torch.save(merged_state_dict)       # 保存为普通 .pth

输出:
  merged_torch_path: ../out/merge_medical_768.pth    # 合并后 (~131 MB)
""")

# 数学验证:merge 后的权重等价于 base + lora
import torch
torch.manual_seed(42)

d = 768
# base 权重
W = torch.randn(d, d) * 0.02
# lora: A (高斯) + B (训练后非零)
A = torch.randn(16, d) * 0.02
B = torch.randn(d, 16) * 0.01  # 假设训练后 B 变成非零

# merge: W' = W + B@A
W_merged = W + B @ A

# 验证:merge 后前向结果等价
x = torch.randn(4, d)
out_base_lora = W @ x.T + (B @ A) @ x.T   # base + lora 分别计算
out_merged = W_merged @ x.T                 # 合并后计算
diff = (out_base_lora - out_merged).abs().max().item()
print(f"merge 误差 (应≈0): {diff:.2e}")
print(f"→ 合并后的单次 matmul 结果与 base+lora 两次 matmul 完全等价")

合并后的 `.pth` 可以直接用 `convert_torch2transformers` 转成 HF 格式,然后部署到 vllm / ollama。

> **关键**:合并是不可逆的。合并后 LoRA 被丢弃,无法再单独更新。所以训练侧保留 base + LoRA 两个文件,部署侧只用合并后的版本。

&nbsp;

---

## Part 4:工具调用评估

### 11.9 eval_toolcall.py:端到端验证

有了 API 服务,怎么验证工具调用功能是否正常?`eval_toolcall.py`(240 行)提供了一套完整的评估框架:

```
用户提问 → 模型生成 <tool_call> → 解析 → 执行 mock 工具 → 结果回传 → 模型继续
```

核心设计:

| 组件 | 代码位置 | 作用 |
|---|---|---|
| `TOOLS` | 第 18-27 行 | 8 个 mock 工具定义 |
| `MOCK_RESULTS` | 第 29-38 行 | 每个工具的 mock 执行函数 |
| `TEST_CASES` | 第 45-54 行 | 8 个测试用例 |
| `parse_tool_calls` | 第 70-78 行 | 从文本提取 tool_call |
| `run_case` | 第 177-199 行 | 多轮循环:生成→调用→回传→再生成 |

In [ ]:
# 展示 eval_toolcall.py 的核心:工具定义和多轮循环
import json

# 8 个 mock 工具 (eval_toolcall.py:18-27 的简化版)
TOOLS = [
    {"type": "function", "function": {"name": "calculate_math",
        "description": "计算数学表达式",
        "parameters": {"type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"]}}},
    {"type": "function", "function": {"name": "get_current_weather",
        "description": "获取天气信息",
        "parameters": {"type": "object",
            "properties": {"location": {"type": "string"}},
            "required": ["location"]}}},
]

# mock 执行 (eval_toolcall.py:29-38 的简化版)
MOCK_RESULTS = {
    "calculate_math": lambda args: {"result": str(eval(args.get("expression", "0")))},
    "get_current_weather": lambda args: {"city": args.get("location"), "temp": "22C"},
}

# 模拟多轮循环 (对应 eval_toolcall.py:177-199 的 run_case)
def run_case_demo(prompt, model_response_fn, tools):
    """模拟: user → assistant(tool_call) → tool(result) → assistant(final)"""
    messages = [{"role": "user", "content": prompt}]
    print(f"💬 [user]: {prompt}\n")

    for turn in range(3):  # 最多 3 轮
        # 模拟模型生成 (真实代码调 model.generate 或 API)
        content, tool_calls = model_response_fn(messages, tools)
        print(f"🧠 [assistant]: {content}")
        if tool_calls:
            print(f"   tool_calls: {json.dumps(tool_calls, ensure_ascii=False)}")

        if not tool_calls:
            break  # 模型不调用工具 → 对话结束

        # 执行工具并回传结果
        messages.append({"role": "assistant", "content": content})
        for tc in tool_calls:
            name = tc["name"]
            args = json.loads(tc["arguments"]) if isinstance(tc["arguments"], str) else tc["arguments"]
            result = MOCK_RESULTS[name](args)
            print(f"📞 [tool {name}]: {json.dumps(result, ensure_ascii=False)}")
            messages.append({"role": "tool", "content": json.dumps(result, ensure_ascii=False)})
        print()

# Mock 模型:第一轮调用 calculate_math,第二轮返回结果
call_count = [0]
def mock_model(messages, tools):
    call_count[0] += 1
    if call_count[0] == 1:
        return "让我算一下", [{"name": "calculate_math", "arguments": '{"expression": "256*37"}'}]
    return "256 × 37 = 9472", None

run_case_demo("帮我算 256 乘以 37", mock_model, TOOLS)

### 评估流程总结

`eval_toolcall.py` 支持两种 backend:
- `--backend local`:直接加载模型,用 `model.generate()` 推理
- `--backend api`:通过 `openai` SDK 调用已部署的 API 服务

```
local backend:                    api backend:
  model.generate()                  client.chat.completions.create()
       ↓                                    ↓
  parse_tool_calls()                response.tool_calls
       ↓                                    ↓
  execute_tool()                    execute_tool()
       ↓                                    ↓
  messages.append(tool result)      messages.append(tool result)
       ↓                                    ↓
  循环直到无 tool_call               循环直到无 tool_call
```

> 两种 backend 共享同一套 `TEST_CASES` 和 `MOCK_RESULTS`,确保评估结果可比。这正好验证了 API 服务的正确性 —— 如果 api backend 和 local backend 结果一致,说明 `serve_openai_api.py` 的实现没问题。

## Summary and takeaways

&nbsp;

---

## 本章总结

### 关键概念回顾

| 概念 | 核心文件:行 | 一句话 |
|---|---|---|
| OpenAI 兼容 API | `serve_openai_api.py:178` | `/v1/chat/completions` 是事实标准 |
| SSE 流式 | `serve_openai_api.py:71-80` | TextStreamer → Queue → `data: {...}\n\n` |
| `<think>` 解析 | `serve_openai_api.py:85-88` | 提取到 `reasoning_content` 字段 |
| `<tool_call>` 解析 | `serve_openai_api.py:93-101` | 提取到 `tool_calls` 字段 |
| open_thinking | `serve_openai_api.py:61-68` | 控制是否生成推理过程 |
| 格式转换 | `convert_model.py:40-96` | `.pth` → HF transformers (Qwen3Config) |
| LoRA 合并 | `convert_model.py:105-112` | $W' = W + BA$,部署用合并版 |
| 工具调用评估 | `eval_toolcall.py` | 8 mock 工具 + 多轮循环 |

### 推理工程的完整链路

```
训练完成
  │
  ├── convert_merge_base_lora (可选:合并 LoRA)
  │
  ├── convert_torch2transformers (.pth → HF Qwen3 格式)
  │       │
  │       ├── vllm serve    (高性能推理)
  │       ├── ollama run    (本地部署)
  │       └── llama.cpp     (CPU / 边缘设备)
  │
  └── serve_openai_api.py (直接部署)
          │
          ├── FastAPI :8998
          ├── /v1/chat/completions (OpenAI 兼容)
          ├── SSE streaming
          ├── <think> → reasoning_content
          └── <tool_call> → tool_calls
                  │
          eval_toolcall.py ← 端到端验证
```

### 三句话总结

1. **协议**:用 OpenAI `/v1/chat/completions` 格式,零成本接入整个生态
2. **流式**:TextStreamer → Queue → SSE,把 TTFT 从「全部完成」降到「第一个 token」
3. **格式**:转换到 HF Qwen3 结构,让 vllm / ollama / llama.cpp 都能加载

> **下一步**:推理工程跑通后,模型已经可以对外服务了。但模型说话不一定讨人喜欢 —— 可能不安全、啰嗦、答非所问。第 12 章开始学习**对齐技术**(DPO / GRPO),让模型更符合人类偏好。
>
> → [第 12 章 · DPO:从偏好学习](../ch12/01_main-chapter-code/README.md)